# Setup

This notebook demonstrates how to generate predictions using MHCflurry.

The setup installs the stable MHCflurry 2.3.0 release and its presentation models.

In [ ]:
# Install the package and download models
%pip install --upgrade "mhcflurry==2.3.2"
!mhcflurry-downloads --quiet fetch models_class1_presentation

In [ ]:
# Imports
import mhcflurry
from google.colab import files

# Quiet warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load a predictor
predictor = mhcflurry.Class1PresentationPredictor.load()
predictor

# Predict for specified peptides

In [ ]:
peptides = """
NLVPMVATV
RANDMPEPTIDE
SIINFEKL
""".split()

alleles = "A*02:01 B*27:01 H2-Kb".split()

results1 = predictor.predict(peptides, alleles)
results1

In [ ]:
# Download results
results1.to_csv('mhcflurry-results.csv')
files.download('mhcflurry-results.csv')

In [ ]:
# See help for more options:
help(predictor.predict)

# Predict by scanning across protein sequences

In [ ]:
# Paste your fasta here
proteins_fasta = """
>tr|A0A6B9WFC7|A0A6B9WFC7_SARS2 Envelope small membrane protein
MYSFVSEETGTLIVNSVLLFLAFVVFLLVTLAILTALRLCAYCCNIVNVSLVKPSFYVYS
RVKNLNSSRVPDLLV
>tr|A0A6B9W0L4|A0A6B9W0L4_SARS2 ORF6 protein
MFHLVDFQVTIAEILLIIMRTFKVSIWNLDYIINLIIKNLSKSLTENKYSQLDEEQPMEI
D
>tr|A0A6G7S6S0|A0A6G7S6S0_SARS2 Nonstructural protein NS3
MDLFMRIFTIGTVTLKQGEIKDATPSDFVRATATIPIQASLPFGWLIVGVALLAVFQSAS
KIITLKKRWQLALSKGVHFVCNLLLLFVTVYSHLLLVAAGLEAPFLYLYALVYFLQSINF
VRIIMRLWLCWKCRSKNPLLYDANYFLCWHTNCYDYCIPYNSVTSSIVITSGDGTTSPIS
EHDYQIGGYTEKWESGVKDCVVLHSYFTSDYYQLYSTQLSTDTGVEHVTFFIYNKIVDEP
EEHVQIHTIDGSSGVVNPVMEPIYDEPTTTTSVPL
>tr|A0A6B9VLF3|A0A6B9VLF3_SARS2 Membrane protein
MADSNGTITVEELKKLLEQWNLVIGFLFLTWICLLQFAYANRNRFLYIIKLIFLWLLWPV
TLACFVLAAVYRINWITGGIAIAMACLVGLMWLSYFIASFRLFARTRSMWSFNPETNILL
NVPLHGTILTRPLLESELVIGAVILRGHLRIAGHHLGRCDIKDLPKEITVATSRTLSYYK
LGASQRVAGDSGFAAYSRYRIGNYKLNTDHSSSSDNIALLVQ
"""

import mhcflurry.fasta

with open("temp.fa", "w") as fd:
    fd.write(proteins_fasta)

proteins = mhcflurry.fasta.read_fasta_to_dataframe("temp.fa").set_index("sequence_id")
proteins

In [ ]:
# Define alleles for each sample
alleles={
    "my-sample": ["A0201", "A0301", "B0702", "C0802"],
}

In [ ]:
# Predict across protein sequences and return peptides with predicted affinity
# less than 500 nM.
results2 = predictor.predict_sequences(
    sequences=proteins.sequence.to_dict(),
    alleles=alleles,
    result="filtered",
    comparison_quantity="affinity",
    filter_value=500)
results2

In [ ]:
# Download results
results2.to_csv('mhcflurry-results.csv')
files.download('mhcflurry-results.csv')

In [ ]:
# See help for more options:
help(predictor.predict_sequences)